In [38]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from io import StringIO
from datetime import date # para pegar a data corrente
from time import sleep



In [39]:
#pip install pandas
#pip install selenium
#pip install lxml


In [40]:


driver = webdriver.Chrome()
driver.get("https://www.fundamentus.com.br/fii_resultado.php")

# esperar a tabela aparecer
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "tabelaResultado"))
)

# pegar o HTML da tabela
tabela = driver.find_element(By.ID, "tabelaResultado")
html_tabela = tabela.get_attribute("outerHTML")

#sleep(10)
driver.quit()



In [41]:

# converter para DataFrame
# fazer o StringIO manter centavos e milhares 
df = pd.read_html(StringIO(html_tabela), decimal=',', thousands='.')[0]

# renomeando as colunas para modo mais amigável
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.normalize('NFKD') # padrão Unicode (Unicode Normalization Form KD) para separar caracteres acentuados "ã" → "a" etc.
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

"""
Papel                object
Segmento             object
Cotação             float64
FFO Yield            object
Dividend Yield       object
P/VP                float64
Valor de Mercado      int64
Liquidez              int64
Qtd de imóveis        int64
Preço do m2         float64
Aluguel por m2      float64
Cap Rate             object
Vacância Média       object
dtype: object
"""

# converte campos percentuais (12,81%) para decimal (0,1281)
for col in df.columns:
    if df[col].astype(str).str.contains('%').any():
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('%', '')
            .str.replace(',', '.')
            .replace('-', '0')
            .astype(float) / 100
        )

# inclui a data corrente, já informando que é datetime como primeira coluna do DF
df.insert(0, 'dia', pd.to_datetime( date.today() ) )


In [42]:

#print(df.dtypes)
#display(df[df['papel'].str.contains('ELDO')])
#display(df)
#df.head(20)


In [43]:
#filtro no modo convencional
"""
df_filtrado = df[
    (df['p_vp'] >= 0.7) & (df['p_vp'] <= 1) &
    (df['dividend_yield'] > 0.07) & (df['dividend_yield'] < 0.12) &
    (df['liquidez'] > 500000)
]
"""

#filtro no formado de query (mais amigável para quem vem do SQL)
df_filtrado = df.query("0.7 < p_vp < 1 and 0.07 < dividend_yield < 0.12 and liquidez > 500000")


In [44]:
df_filtrado.head(100)

,dia,papel,segmento,cotacao,ffo_yield,dividend_yield,p_vp,valor_de_mercado,liquidez,qtd_de_imoveis,preco_do_m2,aluguel_por_m2,cap_rate,vacancia_media
31,2026-05-08,AZPL11,Logística,7.76,0.0713,0.1186,0.90,325320000,617287,1,1580.20,132.78,0.0840,0.0000
41,2026-05-08,BCIA11,Multicategoria,92.42,0.1190,0.1197,0.90,343714000,523161,0,0.00,0.00,0.0000,0.0000
53,2026-05-08,BLMO11,Escritórios,88.49,0.0691,0.0725,0.90,88031100,3328610,1,10675.50,965.29,0.0904,0.0000
69,2026-05-08,BTAL11,Logística,89.93,0.1265,0.0888,0.78,538027000,736331,9,1467.62,186.41,0.1270,0.0000
145,2026-05-08,FGAA11,Outros,9.17,0.1442,0.0886,0.98,413397000,966558,0,0.00,0.00,0.0000,0.0000
169,2026-05-08,GARE11,Multicategoria,8.28,0.0373,0.1191,0.88,2394460000,15838700,3,187.21,27.09,0.1447,0.0000
176,2026-05-08,GGRC11,Multicategoria,10.20,0.0698,0.1169,0.91,2185350000,10525300,41,2237.37,237.34,0.1061,0.0000
198,2026-05-08,HFOF11,Multicategoria,6.71,0.1047,0.1016,0.85,1506600000,2113140,0,0.00,0.00,0.0000,0.0000
201,2026-05-08,HGBS11,Shoppings,20.35,0.0805,0.0921,0.98,2627860000,4887670,17,4452.28,416.55,0.0936,0.0389
206,2026-05-08,HGLG11,Multicategoria,155.98,0.0659,0.0840,0.94,6614280000,15321800,28,3980.92,287.79,0.0723,0.0291
